In [1]:

import cv2
import time
from ultralytics import YOLO

# Load the YOLOv11 model
model = YOLO("yolo12n.pt",task='detect', verbose=False)
# print(model.info()) 

# Load Video
cam = cv2.VideoCapture('Test_Video.mp4')
# cam = cv2.VideoCapture(0) # 0 for webcam

# Class Name Mapping
class_names_dict = model.names

# Classes To Be Detected
class_names_list = ['car','bus','motorcycle']

# Box Color Mapping
color_mapping = {
    # BGR Color Scheme
    0: (0, 0, 255),   # Red for person
    2: (255, 0, 0),   # Blue for car
    3: (255, 255, 0), # Cyan for motorcycle
    5: (0, 255, 0)   # Green for bus
}

# Cross Line Details
y_cross_line = 400

# Initialize Object Count Dictionary
# class_count_dict = {class_name: 0 for class_name in class_names_list}
class_count_dict = {}

# Initialize Set For Object IDs Crossing the Line
crossed_ids = set()

# Get the width and height of the video frame
while cam.isOpened():
    ret, frame = cam.read()
    if not ret:
        # print("Failed to read video frame")
        break
    
    # Run YOLO Object Tracking 
    results = model.track(source=frame,show=False
                          ,persist=True
                          ,classes =[1,2,3,5,7]) # Class ID 1: bicycle, 2: car, 3: motorcycle, 5: bus, 7:truck)
    # print(results)

    # Draw Bounding Boxes and Labels on the Frame
    if (results[0].boxes.data is not None):

        boxes = results[0].boxes

        # # Filter results for specific classes
        # boxes = [i for i in boxes if class_names_dict[i,cls in enumerate(boxes.cls.int().cpu().tolist())] in class_names_list]
        # print(boxes)

        # Extract Object Information
        track_id = boxes.id.int().cpu().tolist()
        conf = boxes.conf.cpu().tolist()
        class_id = boxes.cls.int().cpu().tolist()
        # print(track_id, conf, class_id)

        for box, track_id, conf, class_id in zip(boxes.xyxy, track_id, conf, class_id):
            x1, y1, x2, y2 = map(int, box.cpu().numpy())
            class_label = class_names_dict[class_id]
            label = f"{class_label} {track_id}".capitalize()
            current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

            # Add Camera ID & Time Stamp
            cv2.putText(frame, 'Camera 01', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255),2)
            cv2.putText(frame, current_time, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255),2)

            # Object Detection
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img=frame, text=label, org=(x1, y1 - 10), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, 
                        color=(0, 0, 255), thickness=2)
            
            # Object Tracking Point 
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            cv2.circle(img=frame, center=(cx, cy), radius=10, color=(0, 255, 255), thickness=-1) 

            # Add Crossing Line
            cv2.line(img=frame,pt1=(700,y_cross_line),pt2=(1100, y_cross_line),color=(0, 0, 255),thickness=3)

            # Check if the object crosses the line
            if cy > y_cross_line and track_id not in crossed_ids:
                crossed_ids.add(track_id)
                # Increment the count for the specific class
                if class_label in class_count_dict.keys():
                    # Increment the count for the specific class
                    class_count_dict[class_label] += 1
                else:
                    # Initialize the count for the specific class if not present
                    class_count_dict[class_label] = 1
            
            # Display the count on the frame
            y_offset = 30
            for i in class_count_dict.keys():
                cv2.putText(img=frame, text = f"{i}:{class_count_dict[i]}", org=(10, 50 + y_offset), 
                            fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.75, 
                            color=(0, 255, 0), thickness=2)
                y_offset += 25
    
    # Show Frame
    cv2.imshow('YOLO Object Tracking & Counting', frame)

    # Exit Loop If 'Esc' is Pressed
    # fps = cam.get(cv2.CAP_PROP_FPS)
    # frame_delay = int(1000 / fps)
    # start_time = time.time() # Start time for processing
    # processing_time = (time.time() - start_time) * 1000  # Convert to milliseconds
    # wait_time = max(1, int(frame_delay - processing_time))
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture object and close all OpenCV windows
cam.release()
cv2.destroyAllWindows()

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.9 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 0.9s, installed 1 package: ['lap>=0.5.12']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


0: 384x640 4 cars, 1 bus, 106.5ms
Speed: 5.5ms preprocess, 106.5ms inference, 7.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 bus, 98.0ms
Speed: 1.5ms preprocess, 98.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 bus, 86.1ms
Speed: 1.3ms preprocess, 86.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 bus, 85.0ms
Speed: 1.3ms preprocess, 85.0ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 bus, 86.0ms
Speed: 1.5ms preprocess, 86.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars,

# **Reference**

In [2]:
# https://github.com/AarohiSingla/Vehicle-detection-and-tracking-classwise-using-YOLO11/tree/main
# https://www.youtube.com/watch?v=o8S28sLOUU8